# TGV² objective gap — do we also see the plateau?

*Curiosity notebook.* Reconstructs the **real minimisation objective**

$$F(u,w)=\tfrac12\lVert A u - y\rVert^2+\alpha_1\lVert\nabla u - w\rVert_1+\alpha_2\lVert E w\rVert_1$$

(the exact functional that `pdhg_tomography_tgv.py` minimises), runs the
**zero-deviation** baseline and the **learned** scheme, and plots the gap
$F(x_n)-F^\star$ at every iteration — the same quantity as Fig. 1 of Paper 1.

Goal: check whether the plateau / shoulder appears in the *objective* (not just
in the KKT residual).

## 1. Config

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

SIZE       = 128
N_ANGLES   = 180
SEED       = 1000
NOISE      = 0.02          # match a saved reference if you want (u_ref_0_02.pt)
T          = 500           # iterations to plot (lower this if it's too slow)
CHECKPOINT = "checkpoint_tomo_128_gamma_1.8.pt"   # trained model (gamma ~ 1.89)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 2. Build the problem (same setup as the algorithm)

In [ ]:
from Algo_setuptorch import Params, get_setup, build_algo_functions

params    = Params(size=SIZE)
setup     = get_setup(SIZE, n_angles=N_ANGLES, seed=SEED,
                      noise_level=NOISE, device=device)
functions = build_algo_functions(setup, params)

SHAPES = [(1, 1, SIZE, SIZE),   # u  (image)
          (1, 2, SIZE, SIZE),   # w
          (1, 2, SIZE, SIZE),   # p
          (1, 3, SIZE, SIZE)]   # q

initial_state = setup["initial_state"]
clean         = setup["phantom"]
print(f"gamma = {params.gamma0:.4f}   beta_bar = {params.beta_bar:.3f}")

## 3. The real objective $F(u,w)$

Uses the **normalised** `A` and the normalised sinogram `y` stored by
`get_setup`, with the anisotropic (component-wise) ℓ¹ norms — identical to the
PDHG functional, so the value is directly comparable.

In [ ]:
A    = functions["A"]
y    = functions["data"]
grad = functions["grad"]
E    = functions["E"]
a1, a2 = params.alpha1, params.alpha2

def objective_F(u, w):
    data = 0.5 * (A(u) - y).pow(2).sum()
    reg1 = a1 * (grad(u) - w).abs().sum()   # ||grad u - w||_1
    reg2 = a2 * E(w).abs().sum()            # ||E w||_1
    return float((data + reg1 + reg2).item())

def objective_history(x_hist):
    # x_hist : list of iterates, each [u, w, p, q]
    return np.array([objective_F(x[0], x[1]) for x in x_hist])

## 4. Zero-deviation baseline

In [ ]:
from run import run_zero, run_learned

kkt_zero, res_zero, xhist_zero = run_zero(
    initial_state, functions, params, SHAPES, T=T, device=device)
obj_zero = objective_history(xhist_zero)
print("F_zero:", obj_zero[0], "->", obj_zero[-1])

## 5. Learned scheme

In [ ]:
from algorithm.unrolled_model import UnrolledFBS

model = UnrolledFBS(params=params, shapes=SHAPES, n_channels=3,
                    T=10, alpha=0.99).to(device).float()

ckpt  = torch.load(CHECKPOINT, map_location=device)
state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
model.load_state_dict(state)
model.eval()

kkt_learned, res_learned, hist = run_learned(
    model, initial_state, clean, functions, T_test=T, return_all=True)
obj_learned = objective_history(hist["x"])
print("F_learned:", obj_learned[0], "->", obj_learned[-1])

## 6. Plot $F(x_n)-F^\star$

$F^\star$ is estimated empirically as the lowest objective reached by either run.
The very last points of whichever run attains that minimum will dip to the floor
— that's an artefact of the empirical $F^\star$; the **shoulder/plateau in the
middle** is the thing to look at.

In [ ]:
F_star = min(obj_zero.min(), obj_learned.min())
gap_zero    = np.clip(obj_zero    - F_star, 1e-12, None)
gap_learned = np.clip(obj_learned - F_star, 1e-12, None)

plt.figure(figsize=(6.5, 4.5))
tz = np.arange(1, len(gap_zero) + 1)
tl = np.arange(1, len(gap_learned) + 1)
plt.loglog(tz, gap_zero,    label="Zero (baseline)", lw=2)
plt.loglog(tl, gap_learned, label="Learned",         lw=2, ls="--")

c = gap_zero[0]
plt.loglog(tz, c / tz,      color="gray", lw=1.1, ls="--", label=r"$O(1/n)$")
plt.loglog(tz, c / tz**2,   color="gray", lw=1.1, ls=":",  label=r"$O(1/n^2)$")

plt.xlabel("iteration $n$")
plt.ylabel(r"$F(x_n)-F^\star$")
plt.title("TGV$^2$ objective gap (tomography)")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("plots/objective_gap.pdf", dpi=150)
plt.show()

## 7. (optional) Objective gap vs KKT residual, side by side

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].loglog(tz, gap_zero, lw=2, label="zero")
ax[0].loglog(tl, gap_learned, lw=2, ls="--", label="learned")
ax[0].set_title(r"objective gap $F(x_n)-F^\star$")
ax[0].set_xlabel("n"); ax[0].grid(True, which="both", alpha=0.3); ax[0].legend()

ax[1].loglog(range(1, len(kkt_zero)+1), kkt_zero, lw=2, label="zero")
ax[1].loglog(range(1, len(kkt_learned)+1), kkt_learned, lw=2, ls="--", label="learned")
ax[1].set_title("KKT residual norm")
ax[1].set_xlabel("n"); ax[1].grid(True, which="both", alpha=0.3); ax[1].legend()
plt.tight_layout(); plt.show()